# OCR Metrics Visualization
## So sánh Engine: EasyOCR vs PaddleOCR — Character Accuracy, Edit Distance & Timing

**Biểu đồ:**
1. Character Accuracy (CA) theo thành phần biển số
2. Edit Distance phân bố
3. Inference Time breakdown
4. Radar chart tổng hợp
5. Box-plot so sánh độ chính xác

In [1]:
# ─── Imports ───────────────────────────────────────────────────────────────
import sys, os, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

PROJECT_ROOT = '/Users/binhminh/Project-II'
sys.path.insert(0, os.path.join(PROJECT_ROOT, 'OCR'))

os.makedirs(os.path.join(PROJECT_ROOT, 'OCR', 'output'), exist_ok=True)

print('\u2705 Imports OK')

✅ Imports OK


In [2]:
# ─── 1. Load / Run Pipeline ───────────────────────────────────────────────
import config
from yolo_detector import YoloDetector
from easyocr_engine  import EasyOCREngine
from paddleocr_engine import PaddleOCREngine
from preprocessing import (
    enhance_plate_image,
    postprocess_plate_text,
    parse_plate_components,
)
from metrics import (
    OcrResult, TimingResult,
    levenshtein_distance, normalized_levenshtein,
    component_accuracy, aggregate_confidence,
    compute_aggregate_metrics, print_metrics_report,
)

# Ground truth
gt_csv = os.path.join(PROJECT_ROOT,
                      'data/vietnam-car-license-plate/Bike/GreenParking/location.csv')
df_gt = pd.read_csv(gt_csv, dtype=str).fillna('') if os.path.exists(gt_csv) else None

def load_gt_dict(df_gt):
    if df_gt is None or df_gt.empty:
        return {}
    result = {}
    for _, row in df_gt.iterrows():
        fname = str(row.get('filename', '')).strip()
        if not fname:
            continue
        result[fname] = {
            'full_plate': str(row.get('full_plate', '')).strip(),
            'province':   str(row.get('province',   '')).strip(),
            'series':     str(row.get('series',     '')).strip(),
            'number':     str(row.get('number',     '')).strip(),
        }
    return result

gt_dict = load_gt_dict(df_gt)
print(f'[GT] {len(gt_dict)} ground-truth entries loaded')

Checking connectivity to the model hosters, this may take a while. To bypass this check, set `PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK` to `True`.


ModuleNotFoundError: No module named 'Levenshtein'

In [ ]:
# ─── 2. Helper: process one image ───────────────────────────────────────────
import re, cv2

def _pred_components(pred_str):
    cleaned = re.sub(r'[\s-]', '', str(pred_str).upper())
    province, series, number = '', '', ''
    if len(cleaned) >= 2 and cleaned[:2].isdigit():
        province = cleaned[:2]
        rest = cleaned[2:]
        if len(rest) <= 2:
            series = rest
        else:
            series = rest[:2]
            number = rest[2:]
    else:
        series = cleaned
    return {'province': province, 'series': series, 'number': number}

def _parse_gt_components(plate_str):
    raw = re.sub(r'[\s-]', '', str(plate_str).strip().upper())
    if not raw:
        return {'province': '', 'series': '', 'number': '', 'full': ''}
    province = ''
    series = ''
    number = ''
    if len(raw) >= 2 and raw[:2] in config.PROVINCE_CODES:
        province = raw[:2]
        rest = raw[2:]
    else:
        rest = raw
    if len(rest) <= 2:
        series = rest
    else:
        series = rest[:2]
        number = rest[2:]
    return {'province': province, 'series': series, 'number': number, 'full': province+series+number}

def process_image(image_path, yolo_detector, ocr_engine, engine_name, gt_dict, debug=False):
    image_name = os.path.basename(image_path)
    image = cv2.imread(image_path)
    if image is None:
        return None
    t0 = time.perf_counter()
    detections = yolo_detector.detect(image)
    yolo_time = time.perf_counter() - t0
    if not detections:
        return OcrResult(
            image_name=image_name,
            predicted_plate='NO_PLATE',
            confidence=0.0,
            timing=TimingResult(image_name=image_name, yolo_time=yolo_time, total_time=yolo_time),
            engine=engine_name,
        )
    det = detections[0]
    crop = det['crop']
    enhanced = enhance_plate_image(crop)
    
    if engine_name == 'easyocr':
        raw_data = ocr_engine.readtext(enhanced)
        elapsed = raw_data['elapsed']
        items = raw_data['items']
        if items:
            def avg_y(item):
                bbox = item['bbox']
                return sum(pt[1] for pt in bbox) / len(bbox)
            items_sorted = sorted(items, key=avg_y)
            lines  = [it['text'] for it in items_sorted]
            scores = [it['confidence'] for it in items_sorted]
        else:
            lines, scores = [], []
    else:  # paddleocr
        raw_data = ocr_engine.predict(enhanced)
        elapsed = raw_data['elapsed']
        texts = raw_data['rec_texts']
        polys = raw_data['dt_polys']
        if texts:
            pairs = []
            for i, txt in enumerate(texts):
                poly = polys[i] if i < len(polys) else np.zeros((4, 2))
                y_min = float(np.min(poly[:, 1]))
                pairs.append((y_min, txt))
            pairs.sort(key=lambda x: x[0])
            lines = [t for _, t in pairs]
            scores = raw_data['rec_scores']
        else:
            lines, scores = [], []
    
    ocr_time = elapsed
    final_plate, was_flipped = postprocess_plate_text(lines)
    confidence = aggregate_confidence(scores)
    total_time = yolo_time + ocr_time
    timing = TimingResult(image_name=image_name, yolo_time=yolo_time, ocr_time=ocr_time, total_time=total_time)
    
    # GT evaluation
    gt_full = gt_dict.get(image_name, {}).get('full_plate', '') if gt_dict else ''
    ca_province = ca_series = ca_number = ca_full = False
    ed_full = ed_province = ed_series = ed_number = -1
    ned_full = -1.0
    if gt_full:
        ca_province = component_accuracy(final_plate, gt_full, 'province')
        ca_series   = component_accuracy(final_plate, gt_full, 'series')
        ca_number   = component_accuracy(final_plate, gt_full, 'number')
        ca_full     = component_accuracy(final_plate, gt_full, 'full')
        ed_full     = levenshtein_distance(final_plate, gt_full)
        ned_full    = normalized_levenshtein(final_plate, gt_full)
        gt_parsed   = _parse_gt_components(gt_full)
        pred_comp   = _pred_components(final_plate)
        ed_province = levenshtein_distance(pred_comp['province'], gt_parsed['province']) if gt_parsed['province'] else -1
        ed_series   = levenshtein_distance(pred_comp['series'],   gt_parsed['series'])   if gt_parsed['series']   else -1
        ed_number   = levenshtein_distance(pred_comp['number'],   gt_parsed['number'])   if gt_parsed['number']   else -1
    
    return OcrResult(
        image_name=image_name, predicted_plate=final_plate,
        confidence=confidence, timing=timing,
        ca_province=ca_province, ca_series=ca_series,
        ca_number=ca_number, ca_full=ca_full,
        edit_dist_full=ed_full, norm_edit_dist=ned_full,
        edit_dist_province=ed_province, edit_dist_series=ed_series,
        edit_dist_number=ed_number,
        engine=engine_name, was_flipped=was_flipped,
    )

def run_engine(engine_name, gt_dict, debug=False):
    print(f'\n{\'=\'*50}\n  PIPELINE: YOLO11 + {engine_name.upper()}\n{\'=\'*50}')
    yolo = YoloDetector()
    ocr  = EasyOCREngine() if engine_name == 'easyocr' else PaddleOCREngine()
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp')
    images = [f for f in os.listdir(config.TEST_IMAGE_DIR)
              if f.lower().endswith(valid_exts)]
    images.sort()
    print(f'[Data] {len(images)} images found')
    results = []
    for fname in images:
        r = process_image(os.path.join(config.TEST_IMAGE_DIR, fname),
                         yolo, ocr, engine_name, gt_dict, debug=debug)
        if r:
            results.append(r)
    agg = compute_aggregate_metrics(results)
    print_metrics_report(agg)
    return results, agg

def save_results(results, path):
    rows = []
    for r in results:
        rows.append({
            'filename': r.image_name,
            'predicted_plate': r.predicted_plate,
            'confidence': round(r.confidence, 4),
            'engine': r.engine,
            'was_flipped': r.was_flipped,
            'ca_province': r.ca_province,
            'ca_series': r.ca_series,
            'ca_number': r.ca_number,
            'ca_full': r.ca_full,
            'edit_dist_full': r.edit_dist_full,
            'norm_edit_dist': round(r.norm_edit_dist, 4),
            'edit_dist_province': r.edit_dist_province,
            'edit_dist_series': r.edit_dist_series,
            'edit_dist_number': r.edit_dist_number,
            'yolo_ms': round(r.timing.yolo_ms, 2),
            'ocr_ms': round(r.timing.ocr_ms, 2),
            'total_ms': round(r.timing.total_ms, 2),
        })
    pd.DataFrame(rows).to_csv(path, index=False, encoding='utf-8-sig')
    print(f'[Output] \u2192 {path}')

def load_or_run(force_rerun=False):
    csv_easy   = os.path.join(PROJECT_ROOT, 'OCR', 'output', 'results_easyocr.csv')
    csv_paddle = os.path.join(PROJECT_ROOT, 'OCR', 'output', 'results_paddle.csv')
    
    easy_has   = os.path.exists(csv_easy)
    paddle_has = os.path.exists(csv_paddle)
    
    if (easy_has and paddle_has) and not force_rerun:
        print('[Cache] Loading cached results...')
        df_easy   = pd.read_csv(csv_easy)
        df_paddle = pd.read_csv(csv_paddle)
        agg_easy   = compute_aggregate_metrics([_row_to_result(row) for _, row in df_easy.iterrows()])
        agg_paddle = compute_aggregate_metrics([_row_to_result(row) for _, row in df_paddle.iterrows()])
        return df_easy, df_paddle, agg_easy, agg_paddle
    
    # Run pipelines
    res_easy, agg_easy   = run_engine('easyocr',   gt_dict, debug=False)
    res_paddle, agg_paddle = run_engine('paddleocr', gt_dict, debug=False)
    
    save_results(res_easy,   csv_easy)
    save_results(res_paddle, csv_paddle)
    
    return res_easy, res_paddle, agg_easy, agg_paddle

def _row_to_result(row):
    tr = TimingResult(image_name=row['filename'],
                      yolo_time=row['yolo_ms']/1000,
                      ocr_time=row['ocr_ms']/1000,
                      total_time=row['total_ms']/1000)
    return OcrResult(
        image_name=row['filename'],
        predicted_plate=row['predicted_plate'],
        confidence=row['confidence'],
        timing=tr,
        ca_province=row.get('ca_province',False),
        ca_series=row.get('ca_series',False),
        ca_number=row.get('ca_number',False),
        ca_full=row.get('ca_full',False),
        edit_dist_full=row.get('edit_dist_full',-1),
        norm_edit_dist=row.get('norm_edit_dist',-1.0),
        edit_dist_province=row.get('edit_dist_province',-1),
        edit_dist_series=row.get('edit_dist_series',-1),
        edit_dist_number=row.get('edit_dist_number',-1),
        engine=row.get('engine','unknown'),
        was_flipped=row.get('was_flipped',False),
    )

print('\u2705 Helper functions defined')

In [ ]:
# ─── 3. Run (or load cached) results ────────────────────────────────────────
# Bỏ comment dòng dưới để chạy lại pipeline từ đầu
# results_easy, results_paddle, agg_easy, agg_paddle = load_or_run(force_rerun=True)
results_easy, results_paddle, agg_easy, agg_paddle = load_or_run(force_rerun=False)

In [ ]:
# ─── 4. Build comparison DataFrame ──────────────────────────────────────────
def build_comparison_df(df_easy, df_paddle):
    """Ghép 2 engine vào 1 df để so sánh per-image."""
    df_easy   = df_easy.copy().rename(columns={
        c: f'{c}_easy' for c in df_easy.columns if c != 'filename'
    })
    df_paddle = df_paddle.copy().rename(columns={
        c: f'{c}_paddle' for c in df_paddle.columns if c != 'filename'
    })
    merged = df_easy.merge(df_paddle, on='filename', how='outer')
    return merged

df_cmp = build_comparison_df(results_easy, results_paddle)

# Summary table
summary = pd.DataFrame({
    'Metric': [
        'CA Province (%)', 'CA Series (%)', 'CA Number (%)', 'CA Full (%)',
        'Mean Edit Distance', 'Norm Edit Dist',
        'Mean Confidence', 'Flipped Fixed',
        'Mean Total (ms)', 'Mean YOLO (ms)', 'Mean OCR (ms)',
        'N Images',
    ],
    'EasyOCR': [
        agg_easy.get('ca_province_%', 0), agg_easy.get('ca_series_%', 0),
        agg_easy.get('ca_number_%', 0),  agg_easy.get('ca_full_%', 0),
        agg_easy.get('mean_edit_dist', 0), '-',
        agg_easy.get('mean_confidence', 0), agg_easy.get('flipped_count', 0),
        agg_easy.get('mean_total_ms', 0), '-', agg_easy.get('mean_ocr_ms', 0),
        agg_easy.get('n_images', 0),
    ],
    'PaddleOCR': [
        agg_paddle.get('ca_province_%', 0), agg_paddle.get('ca_series_%', 0),
        agg_paddle.get('ca_number_%', 0),  agg_paddle.get('ca_full_%', 0),
        agg_paddle.get('mean_edit_dist', 0), '-',
        agg_paddle.get('mean_confidence', 0), agg_paddle.get('flipped_count', 0),
        agg_paddle.get('mean_total_ms', 0), '-', agg_paddle.get('mean_ocr_ms', 0),
        agg_paddle.get('n_images', 0),
    ],
})
summary['Winner'] = summary.apply(
    lambda r: 'EasyOCR' if isinstance(r['EasyOCR'], (int, float)) and isinstance(r['PaddleOCR'], (int, float))
               and (r['Metric'] not in ['Mean Total (ms)', 'Mean OCR (ms)', 'Mean YOLO (ms)']
                    and r['EasyOCR'] > r['PaddleOCR']
                    or r['Metric'] in ['Mean Total (ms)', 'Mean OCR (ms)', 'Mean YOLO (ms)']
                    and r['EasyOCR'] < r['PaddleOCR'])
               else ('PaddleOCR' if isinstance(r['EasyOCR'], (int, float)) and isinstance(r['PaddleOCR'], (int, float))
                     else '='), axis=1
)

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)
print('\n\u2572\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573')
print('                         ENGINE COMPARISON SUMMARY')
print('\u2572\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573\u2573')
print(summary.to_string(index=False))

In [ ]:
# ─── 5. Style helpers ───────────────────────────────────────────────────────
ENGINES = ['EasyOCR', 'PaddleOCR']
COLORS  = {'EasyOCR': '#4C78A8', 'PaddleOCR': '#E45756'}
ACC_COLORS = {'EasyOCR': '#3A9BD5', 'PaddleOCR': '#F4A261'}

plt.rcParams.update({
    'figure.facecolor':  '#1A1A2E',
    'axes.facecolor':    '#16213E',
    'axes.edgecolor':    '#30475E',
    'axes.labelcolor':   '#E0E0E0',
    'xtick.color':       '#A0A0A0',
    'ytick.color':       '#A0A0A0',
    'text.color':        '#E0E0E0',
    'grid.color':        '#30475E',
    'grid.alpha':        0.5,
    'axes.titlesize':    13,
    'axes.labelsize':    11,
    'legend.fontsize':   10,
    'font.family':       'DejaVu Sans',
})

In [ ]:
# ─── 6. FIGURE 1 – Character Accuracy (CA) Bar Chart ─────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Character Accuracy (CA) by Plate Component', fontsize=16, fontweight='bold', y=1.02)

components = ['Province', 'Series', 'Number', 'Full']
easy_vals   = [agg_easy.get(f'ca_{c.lower()}_%', 0)   for c in components]
paddle_vals = [agg_paddle.get(f'ca_{c.lower()}_%', 0) for c in components]

# Left: grouped bar
x = np.arange(len(components))
w = 0.35
bars_e = axes[0].bar(x - w/2, easy_vals,   w, label='EasyOCR',   color=COLORS['EasyOCR'],   edgecolor='white', linewidth=0.5)
bars_p = axes[0].bar(x + w/2, paddle_vals, w, label='PaddleOCR', color=COLORS['PaddleOCR'], edgecolor='white', linewidth=0.5)

for bar, val in zip(bars_e, easy_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, color='#E0E0E0')
for bar, val in zip(bars_p, paddle_vals):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 f'{val:.1f}%', ha='center', va='bottom', fontsize=9, color='#E0E0E0')

axes[0].set_xticks(x)
axes[0].set_xticklabels(components, fontsize=11)
axes[0].set_ylabel('Accuracy (%)')
axes[0].set_ylim(0, max(max(easy_vals), max(paddle_vals)) * 1.15 + 5)
axes[0].set_title('Grouped Comparison', fontsize=12)
axes[0].legend(loc='upper left')
axes[0].grid(axis='y', alpha=0.4)

# Right: radar-style 4-axis spider
angles = np.linspace(0, 2*np.pi, len(components), endpoint=False).tolist()
angles += angles[:1]

ax_radar = axes[1] if False else fig.add_subplot(122, projection='polar')

for name, vals in [('EasyOCR', easy_vals), ('PaddleOCR', paddle_vals)]:
    v = vals + vals[:1]
    ax_radar.plot(angles, v, 'o-', linewidth=2, color=COLORS[name], label=name)
    ax_radar.fill(angles, v, alpha=0.15, color=COLORS[name])

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(components, fontsize=10)
ax_radar.set_ylim(0, 105)
ax_radar.set_title('Radar – CA Coverage', fontsize=12, pad=15)
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'OCR', 'output', 'fig1_ca_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 7. FIGURE 2 – Edit Distance Distribution ───────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Edit Distance (Levenshtein) Distribution per Image', fontsize=16, fontweight='bold')

ed_easy   = results_easy.edit_dist_full.values if hasattr(results_easy, 'edit_dist_full') else []
ed_paddle = results_paddle.edit_dist_full.values if hasattr(results_paddle, 'edit_dist_full') else []

ed_easy   = np.array([v for v in ed_easy   if v >= 0])
ed_paddle = np.array([v for v in ed_paddle if v >= 0])

# Histogram
bins = np.arange(0, max(ed_easy.max() if len(ed_easy) else 10,
                         ed_paddle.max() if len(ed_paddle) else 10) + 2)
axes[0].hist(ed_easy,   bins=bins, alpha=0.7, label='EasyOCR',   color=COLORS['EasyOCR'],   edgecolor='white')
axes[0].hist(ed_paddle, bins=bins, alpha=0.7, label='PaddleOCR', color=COLORS['PaddleOCR'], edgecolor='white')
axes[0].axvline(ed_easy.mean(),   color=COLORS['EasyOCR'],   linestyle='--', linewidth=2, label=f'Easy mean={ed_easy.mean():.2f}')
axes[0].axvline(ed_paddle.mean(), color=COLORS['PaddleOCR'], linestyle='--', linewidth=2, label=f'Paddle mean={ed_paddle.mean():.2f}')
axes[0].set_xlabel('Edit Distance')
axes[0].set_ylabel('Count')
axes[0].set_title('Histogram')
axes[0].legend(fontsize=9)
axes[0].grid(axis='y', alpha=0.4)

# Box plot
bp = axes[1].boxplot([ed_easy, ed_paddle],
                     labels=ENGINES,
                     patch_artist=True,
                     boxprops=dict(linewidth=1.5),
                     medianprops=dict(color='yellow', linewidth=2))
for patch, name in zip(bp['boxes'], ENGINES):
    patch.set_facecolor(COLORS[name])
    patch.set_alpha(0.7)
axes[1].set_ylabel('Edit Distance')
axes[1].set_title('Box Plot')
axes[1].grid(axis='y', alpha=0.4)

# CDF
def cdf(data):
    s = np.sort(data)
    return s, np.arange(1, len(s)+1) / len(s)
if len(ed_easy):
    sx, cy = cdf(ed_easy)
    axes[2].plot(sx, cy, color=COLORS['EasyOCR'], linewidth=2, label='EasyOCR')
if len(ed_paddle):
    sx, cy = cdf(ed_paddle)
    axes[2].plot(sx, cy, color=COLORS['PaddleOCR'], linewidth=2, label='PaddleOCR')
axes[2].set_xlabel('Edit Distance')
axes[2].set_ylabel('Cumulative Probability')
axes[2].set_title('CDF')
axes[2].legend(fontsize=10)
axes[2].grid(alpha=0.4)
axes[2].set_ylim(0, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'OCR', 'output', 'fig2_edit_distance.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 8. FIGURE 3 – Timing Breakdown ──────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle('Inference Time Breakdown', fontsize=16, fontweight='bold')

yolo_easy   = results_easy.yolo_ms.values   if hasattr(results_easy, 'yolo_ms')   else []
ocr_easy    = results_easy.ocr_ms.values    if hasattr(results_easy, 'ocr_ms')   else []
total_easy  = results_easy.total_ms.values if hasattr(results_easy, 'total_ms') else []
yolo_paddle   = results_paddle.yolo_ms.values   if hasattr(results_paddle, 'yolo_ms')   else []
ocr_paddle    = results_paddle.ocr_ms.values    if hasattr(results_paddle, 'ocr_ms')   else []
total_paddle  = results_paddle.total_ms.values if hasattr(results_paddle, 'total_ms') else []

yolo_easy   = np.array([v for v in yolo_easy   if v >= 0])
ocr_easy    = np.array([v for v in ocr_easy    if v >= 0])
total_easy  = np.array([v for v in total_easy  if v >= 0])
yolo_paddle   = np.array([v for v in yolo_paddle   if v >= 0])
ocr_paddle    = np.array([v for v in ocr_paddle    if v >= 0])
total_paddle  = np.array([v for v in total_paddle  if v >= 0])

mean_labels = ['YOLO Detection', 'OCR Inference', 'Total']
easy_means   = [yolo_easy.mean(),   ocr_easy.mean(),   total_easy.mean()]
paddle_means = [yolo_paddle.mean(), ocr_paddle.mean(), total_paddle.mean()]

# Bar: stacked mean per image
x = np.arange(len(mean_labels))
w = 0.35
axes[0].bar(x - w/2, easy_means,   w, label='EasyOCR',   color=COLORS['EasyOCR'],   edgecolor='white')
axes[0].bar(x + w/2, paddle_means, w, label='PaddleOCR', color=COLORS['PaddleOCR'], edgecolor='white')
for xi, ev, pv in zip(x, easy_means, paddle_means):
    axes[0].text(xi - w/2, ev + 1, f'{ev:.1f}ms', ha='center', va='bottom', fontsize=8, color='#E0E0E0')
    axes[0].text(xi + w/2, pv + 1, f'{pv:.1f}ms', ha='center', va='bottom', fontsize=8, color='#E0E0E0')
axes[0].set_xticks(x)
axes[0].set_xticklabels(mean_labels)
axes[0].set_ylabel('Time (ms)')
axes[0].set_title('Mean per Image')
axes[0].legend()
axes[0].grid(axis='y', alpha=0.4)

# Histogram of total time
axes[1].hist(total_easy,   30, alpha=0.6, label='EasyOCR',   color=COLORS['EasyOCR'],   edgecolor='white')
axes[1].hist(total_paddle, 30, alpha=0.6, label='PaddleOCR', color=COLORS['PaddleOCR'], edgecolor='white')
axes[1].axvline(total_easy.mean(),   color=COLORS['EasyOCR'],   linestyle='--', linewidth=2)
axes[1].axvline(total_paddle.mean(), color=COLORS['PaddleOCR'], linestyle='--', linewidth=2)
axes[1].set_xlabel('Total Time (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Total Time Distribution')
axes[1].legend()
axes[1].grid(axis='y', alpha=0.4)

# Pie: time split per engine
sizes_e = [yolo_easy.mean(), ocr_easy.mean()]
sizes_p = [yolo_paddle.mean(), ocr_paddle.mean()]
labels  = ['YOLO', 'OCR']

ax_pie1 = fig.add_subplot(2, 3, 4, aspect='auto')
ax_pie2 = fig.add_subplot(2, 3, 5, aspect='auto')

wedges, texts, autotexts = ax_pie1.pie(sizes_e, labels=labels, autopct='%1.1f%%',
                                       colors=[COLORS['EasyOCR'], '#7EC8E3'],
                                       textprops={'color': 'white'}, autopctprops={'color': 'white'})
ax_pie1.set_title('EasyOCR Time Split', fontsize=11)

wedges, texts, autotexts = ax_pie2.pie(sizes_p, labels=labels, autopct='%1.1f%%',
                                       colors=[COLORS['PaddleOCR'], '#7EC8E3'],
                                       textprops={'color': 'white'}, autopctprops={'color': 'white'})
ax_pie2.set_title('PaddleOCR Time Split', fontsize=11)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'OCR', 'output', 'fig3_timing.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 9. FIGURE 4 – Radar Chart Tổng Hợp ─────────────────────────────────────
metrics_radar = {
    'CA Full (%)':       (agg_easy.get('ca_full_%', 0),        agg_paddle.get('ca_full_%', 0)),
    'CA Province (%)':   (agg_easy.get('ca_province_%', 0),     agg_paddle.get('ca_province_%', 0)),
    'CA Number (%)':    (agg_easy.get('ca_number_%', 0),       agg_paddle.get('ca_number_%', 0)),
    'CA Series (%)':    (agg_easy.get('ca_series_%', 0),       agg_paddle.get('ca_series_%', 0)),
    'Confidence':       (agg_easy.get('mean_confidence', 0)*100, agg_paddle.get('mean_confidence', 0)*100),
    # Speed: invert so higher = better
    'Speed (ms)':       (100/max(agg_easy.get('mean_total_ms',1),1),   100/max(agg_paddle.get('mean_total_ms',1),1)),
    # Lower edit dist = better, normalize 0-100
    'Accuracy (low ED)': (100/(agg_easy.get('mean_edit_dist',0.5)+1),  100/(agg_paddle.get('mean_edit_dist',0.5)+1)),
}

labels = list(metrics_radar.keys())
easy_vals   = [v[0] for v in metrics_radar.values()]
paddle_vals = [v[1] for v in metrics_radar.values()]

N = len(labels)
angles = np.linspace(0, 2*np.pi, N, endpoint=False).tolist()
angles += angles[:1]
easy_vals   += easy_vals[:1]
paddle_vals += paddle_vals[:1]

fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
fig.patch.set_facecolor('#1A1A2E')

ax.plot(angles, easy_vals,   'o-', linewidth=2.5, color=COLORS['EasyOCR'],   label='EasyOCR',   markersize=6)
ax.fill(angles, easy_vals,   alpha=0.20, color=COLORS['EasyOCR'])
ax.plot(angles, paddle_vals, 'o-', linewidth=2.5, color=COLORS['PaddleOCR'], label='PaddleOCR', markersize=6)
ax.fill(angles, paddle_vals, alpha=0.20, color=COLORS['PaddleOCR'])

ax.set_xticks(angles[:-1])
ax.set_xticklabels(labels, fontsize=10, color='#E0E0E0')
ax.set_ylim(0, 110)
ax.set_title('Radar – Engine Comparison (All Metrics)', fontsize=14, fontweight='bold', pad=20, color='#E0E0E0')
ax.legend(loc='upper right', bbox_to_anchor=(1.35, 1.1), fontsize=11)
ax.grid(color='#30475E', alpha=0.6)
ax.tick_params(colors='#A0A0A0')

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'OCR', 'output', 'fig4_radar_all.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 10. FIGURE 5 – Box Plot: Confidence & Norm Edit Dist ───────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Confidence & Normalized Edit Distance per Engine', fontsize=14, fontweight='bold')

conf_easy   = results_easy.confidence.values   if hasattr(results_easy, 'confidence')   else []
conf_paddle = results_paddle.confidence.values if hasattr(results_paddle, 'confidence') else []
ned_easy   = results_easy.norm_edit_dist.values   if hasattr(results_easy, 'norm_edit_dist')   else []
ned_paddle = results_paddle.norm_edit_dist.values if hasattr(results_paddle, 'norm_edit_dist') else []

ned_easy   = np.array([v for v in ned_easy   if v >= 0])
ned_paddle = np.array([v for v in ned_paddle if v >= 0])

bp1 = axes[0].boxplot([conf_easy, conf_paddle], labels=ENGINES,
                      patch_artist=True,
                      boxprops=dict(linewidth=1.5),
                      medianprops=dict(color='#FFD700', linewidth=2),
                      whiskerprops=dict(color='white'),
                      capprops=dict(color='white'))
for patch, name in zip(bp1['boxes'], ENGINES):
    patch.set_facecolor(COLORS[name])
    patch.set_alpha(0.7)
axes[0].set_ylabel('Confidence Score')
axes[0].set_title('Confidence Distribution')
axes[0].grid(axis='y', alpha=0.4)
axes[0].set_ylim(0, 1.05)

bp2 = axes[1].boxplot([ned_easy, ned_paddle], labels=ENGINES,
                      patch_artist=True,
                      boxprops=dict(linewidth=1.5),
                      medianprops=dict(color='#FFD700', linewidth=2),
                      whiskerprops=dict(color='white'),
                      capprops=dict(color='white'))
for patch, name in zip(bp2['boxes'], ENGINES):
    patch.set_facecolor(COLORS[name])
    patch.set_alpha(0.7)
axes[1].set_ylabel('Normalized Edit Distance (0-1)')
axes[1].set_title('Normalized Edit Distance')
axes[1].grid(axis='y', alpha=0.4)
axes[1].set_ylim(-0.02, 1.05)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'OCR', 'output', 'fig5_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 11. FIGURE 6 – Per-image scatter: EasyOCR vs PaddleOCR ─────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Per-Image Comparison: EasyOCR vs PaddleOCR', fontsize=14, fontweight='bold')

conf_easy_arr   = results_easy.confidence.values   if hasattr(results_easy, 'confidence')   else []
conf_paddle_arr = results_paddle.confidence.values if hasattr(results_paddle, 'confidence') else []

ed_easy_arr   = results_easy.edit_dist_full.values   if hasattr(results_easy, 'edit_dist_full')   else []
ed_paddle_arr = results_paddle.edit_dist_full.values if hasattr(results_paddle, 'edit_dist_full') else []

ed_easy_arr   = np.array([v for v in ed_easy_arr   if v >= 0])
ed_paddle_arr = np.array([v for v in ed_paddle_arr if v >= 0])

n = min(len(conf_easy_arr), len(conf_paddle_arr))
if n > 0:
    axes[0].scatter(conf_easy_arr[:n], conf_paddle_arr[:n],
                   alpha=0.4, c=COLORS['EasyOCR'], s=20, label='Images')
    max_c = max(conf_easy_arr[:n].max(), conf_paddle_arr[:n].max())
    axes[0].plot([0, max_c], [0, max_c], 'w--', linewidth=1, label='y=x (tie)')
    axes[0].set_xlabel('EasyOCR Confidence')
    axes[0].set_ylabel('PaddleOCR Confidence')
    axes[0].set_title('Confidence per Image')
    axes[0].legend()
    axes[0].grid(alpha=0.4)
    axes[0].set_xlim(0, 1.05)
    axes[0].set_ylim(0, 1.05)

n_ed = min(len(ed_easy_arr), len(ed_paddle_arr))
if n_ed > 0:
    axes[1].scatter(ed_easy_arr[:n_ed], ed_paddle_arr[:n_ed],
                    alpha=0.4, c=COLORS['PaddleOCR'], s=20, label='Images')
    max_ed = max(ed_easy_arr[:n_ed].max(), ed_paddle_arr[:n_ed].max())
    axes[1].plot([0, max_ed], [0, max_ed], 'w--', linewidth=1, label='y=x (tie)')
    axes[1].set_xlabel('EasyOCR Edit Distance')
    axes[1].set_ylabel('PaddleOCR Edit Distance')
    axes[1].set_title('Edit Distance per Image')
    axes[1].legend()
    axes[1].grid(alpha=0.4)

plt.tight_layout()
plt.savefig(os.path.join(PROJECT_ROOT, 'OCR', 'output', 'fig6_scatter.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ─── 12. Export Final Summary CSV ─────────────────────────────────────────
summary_path = os.path.join(PROJECT_ROOT, 'OCR', 'output', 'comparison_summary.csv')
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
print(f'[Export] \u2192 {summary_path}')

print('\n\u2705 All figures saved to OCR/output/')
print('  fig1_ca_comparison.png')
print('  fig2_edit_distance.png')
print('  fig3_timing.png')
print('  fig4_radar_all.png')
print('  fig5_boxplot.png')
print('  fig6_scatter.png')